<a href="https://colab.research.google.com/github/ancestor9/Listeria/blob/main/timeseries_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **A. Regression**

In [38]:
# @title 1. 단일 모델 직접(direct) 다중 호라이즌 예측
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# 1. 시뮬레이션용 예제 데이터 생성 (100일 치)
np.random.seed(42)
dates = pd.date_range(start="2024-01-01", periods=100, freq="D")
y_values = 20 + np.sin(np.linspace(0, 10, 100)) * 5 + np.random.normal(0, 1, 100)
df = pd.DataFrame({"ds": dates, "y": y_values})
df.head(20)

,ds,y
0,2024-01-01,20.496714
1,2024-01-02,20.365928
2,2024-01-03,21.650933
3,2024-01-04,23.015099
4,2024-01-05,21.731530
5,2024-01-06,22.185121
6,2024-01-07,24.427383
7,2024-01-08,24.015482
8,2024-01-09,23.145338
9,2024-01-10,24.487287


In [20]:
df.tail()

,ds,y
95,2024-04-05,17.684751
96,2024-04-06,18.951905
97,2024-04-07,18.438062
98,2024-04-08,17.721926
99,2024-04-09,17.045307


In [25]:
# 2. 파라미터 설정 (과거 6일 데이터로 향후 3일 예측)
WINDOW_SIZE = 6  # 과거 6일 (Window = 6)
MAX_H = 3        # 향후 3일 (Forecast Horizon H = 1, 2, 3)

X_list = []
y_list = []

# t는 기준일(D)의 인덱스
n_days = len(df)
for t in range(WINDOW_SIZE - 1, n_days - MAX_H):
    # 기준일(t) 이전 과거 6일간의 실측 데이터 추출
    window_data = df["y"].iloc[t - WINDOW_SIZE + 1 : t + 1].values

    # 과거 6일 데이터의 요약 피처 생성
    mean_val = np.mean(window_data)
    std_val = np.std(window_data)
    min_val = np.min(window_data)
    max_val = np.max(window_data)
    last_val = window_data[-1]  # 기준일(D) 당일 실측값

    # H = 1, 2, 3에 대해 각각 데이터행 구성
    for h in range(1, MAX_H + 1):
        target = df["y"].iloc[t + h]  # t+H 시점의 실제 정답값

        # 입력 피처: [과거 6일 요약 피처들 + H]
        feature_vector = [mean_val, std_val, min_val, max_val, last_val, h]
        X_list.append(feature_vector)
        y_list.append(target)

X = np.array(X_list)
y = np.array(y_list)
X[:9]

array([[21.57422076,  0.92204052, 20.3659278 , 23.01509888, 22.18512125,
         1.        ],
       [21.57422076,  0.92204052, 20.3659278 , 23.01509888, 22.18512125,
         2.        ],
       [21.57422076,  0.92204052, 20.3659278 , 23.01509888, 22.18512125,
         3.        ],
       [22.2293323 ,  1.25866609, 20.3659278 , 24.42738335, 24.42738335,
         1.        ],
       [22.2293323 ,  1.25866609, 20.3659278 , 24.42738335, 24.42738335,
         2.        ],
       [22.2293323 ,  1.25866609, 20.3659278 , 24.42738335, 24.42738335,
         3.        ],
       [22.83759138,  1.08040291, 21.65093282, 24.42738335, 24.0154823 ,
         1.        ],
       [22.83759138,  1.08040291, 21.65093282, 24.42738335, 24.0154823 ,
         2.        ],
       [22.83759138,  1.08040291, 21.65093282, 24.42738335, 24.0154823 ,
         3.        ]])

In [22]:
y[:10]

array([24.42738335, 24.0154823 , 23.14533842, 24.0154823 , 23.14533842,
       24.48728736, 23.14533842, 24.48728736, 23.77101013, 24.48728736])

In [24]:
y[-9:]

array([17.68475089, 18.95190465, 18.4380616 , 18.95190465, 18.4380616 ,
       17.72192602, 18.4380616 , 17.72192602, 17.04530731])

In [21]:
# 3. 단일 예측 모델 학습
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)
model

RandomForestRegressor(n_estimators=50, random_state=42)

df["ds"] == "2024-03-01" 조건으로 찾은 위치 숫자가 60이었기 때문에, 모델은 0번부터 59번까지(과거 6일 데이터는 55~60번)의 데이터를 바탕으로 61번(3월 2일), 62번(3월 3일), 63번(3월 4일)을 예측하게 되는 메커니즘

In [26]:
# 4. 특정 기준일(D = 2024-03-01) 설정 후 D+1 ~ D+3 예측 수행
ref_idx = df[df["ds"] == "2024-03-01"].index[0]
ref_window = df["y"].iloc[ref_idx - WINDOW_SIZE + 1 : ref_idx + 1].values

# 기준일 D 시점의 과거 6일 요약 피처 (고정)
ref_features = [
    np.mean(ref_window),
    np.std(ref_window),
    np.min(ref_window),
    np.max(ref_window),
    ref_window[-1]
]
ref_idx

np.int64(60)

In [27]:
# H = 1, 2, 3 변수만 바꿔가며 동일한 모델에서 예측치 산출
predictions = []
for h in range(1, MAX_H + 1):
    input_feat = [ref_features + [h]]  # 과거 6일 피처 고정 + H(1,2,3) 입력
    pred = model.predict(input_feat)[0]
    predictions.append(pred)


# 5. 결과 확인
pred_df = pd.DataFrame({
    "Forecast_Date": [df["ds"].iloc[ref_idx] + pd.Timedelta(days=h) for h in range(1, MAX_H + 1)],
    "H": range(1, MAX_H + 1),
    "Predicted_Y": predictions,
    "Actual_Y": df["y"].iloc[ref_idx + 1 : ref_idx + MAX_H + 1].values
})

pred_df

,Forecast_Date,H,Predicted_Y,Actual_Y
0,2024-03-02,1,19.083052,19.207991
1,2024-03-03,2,18.871847,18.790877
2,2024-03-04,3,19.218761,19.205615


In [40]:
# @title 2. 예측일 미래 3일 벡터 타깃
X_list = []
y_list = []

for t in range(WINDOW_SIZE - 1, n_days - MAX_H):
    window_data = df["y"].iloc[t - WINDOW_SIZE + 1 : t + 1].values

    feature_vector = [
        np.mean(window_data), np.std(window_data),
        np.min(window_data), np.max(window_data),
        window_data[-1],
    ]                                            # H 없음 (5개 피처)
    target_vector = df["y"].iloc[t + 1 : t + MAX_H + 1].values  # 길이 3

    X_list.append(feature_vector)
    y_list.append(target_vector)

X = np.array(X_list)   # (N, 5)
y = np.array(y_list)   # (N, 3)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (92, 5)
y shape: (92, 3)


In [32]:
X[:9]

array([[21.57422076,  0.92204052, 20.3659278 , 23.01509888, 22.18512125],
       [22.2293323 ,  1.25866609, 20.3659278 , 24.42738335, 24.42738335],
       [22.83759138,  1.08040291, 21.65093282, 24.42738335, 24.0154823 ],
       [23.08665898,  0.94144963, 21.73152969, 24.42738335, 23.14533842],
       [23.33202373,  1.07341979, 21.73152969, 24.48728736, 24.48728736],
       [23.67193713,  0.8011722 , 22.18512125, 24.48728736, 23.77101013],
       [23.97695547,  0.44726459, 23.14533842, 24.48728736, 24.01523125],
       [24.05968756,  0.55569622, 23.14533842, 24.9237759 , 24.9237759 ],
       [23.87738349,  0.70077797, 22.92165787, 24.9237759 , 22.92165787]])

In [33]:
y[:9]

array([[24.42738335, 24.0154823 , 23.14533842],
       [24.0154823 , 23.14533842, 24.48728736],
       [23.14533842, 24.48728736, 23.77101013],
       [24.48728736, 23.77101013, 24.01523125],
       [23.77101013, 24.01523125, 24.9237759 ],
       [24.01523125, 24.9237759 , 22.92165787],
       [24.9237759 , 22.92165787, 23.21385563],
       [22.92165787, 23.21385563, 24.42997361],
       [23.21385563, 24.42997361, 23.98202474]])

In [35]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)

RandomForestRegressor(n_estimators=50, random_state=42)

In [37]:
predictions = model.predict([ref_features])[0]   # [D+1, D+2, D+3] 예측값
predictions

array([18.95763535, 18.85717589, 19.06248559])

## **B. Binary Classification**

In [41]:
"""
100일 시계열 -> 과거 6일로 향후 3일(H=1,2,3)의 이진 이벤트 예측
  방식 A: 단일 모델 직접(direct) 다중 호라이즌 (H를 피처로 투입)
  방식 B: multi-label 벡터 타깃 (한 행 = 기준일, 타깃 = 3개 0/1)
학습: 앞 70일 / 테스트: 뒤 30일 (시간 순서 분할, 누수 방지)
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, brier_score_loss)

# ------------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------------
SEED = 42
N_DAYS = 100        # 전체 기간
WINDOW = 6          # 입력: 과거 6일
MAX_H = 3           # 예측: 향후 3일 (H = 1, 2, 3)
TRAIN_DAYS = 70     # 0~69일 학습, 70~99일 테스트
EVENT_THR = 21.0    # y가 이 값을 넘으면 이벤트(1)
np.random.seed(SEED)

# ------------------------------------------------------------------
# 1. 100일 시계열 생성 + 이진 이벤트 라벨
# ------------------------------------------------------------------
t_axis = np.arange(N_DAYS)
y = 20 + 5 * np.sin(2 * np.pi * t_axis / 20) + np.random.normal(0, 1, N_DAYS)
event = (y > EVENT_THR).astype(int)   # 하루 단위 이진 라벨 (시점형 타깃)
print(f"전체 양성 비율: {event.mean():.2f}")


# ------------------------------------------------------------------
# 2. 피처/타깃 생성 함수
# ------------------------------------------------------------------
def window_features(win):
    """과거 6일 윈도우 -> 요약 피처 6개 (평균, 표준편차, 최소, 최대, 마지막값, 기울기)"""
    slope = np.polyfit(np.arange(len(win)), win, 1)[0]   # 추세 정보
    return [win.mean(), win.std(), win.min(), win.max(), win[-1], slope]


def build_samples(t_list):
    """
    기준일 t마다 1행 생성.
      X    : (n, 6)      기준일 t 이전 6일 요약 피처
      Yvec : (n, MAX_H)  [event[t+1], event[t+2], event[t+3]]
    """
    X = np.array([window_features(y[t - WINDOW + 1: t + 1]) for t in t_list])
    Yvec = np.array([event[t + 1: t + MAX_H + 1] for t in t_list])
    return X, Yvec


# ------------------------------------------------------------------
# 3. 시간 순서 분할 (기준일 t 단위로 분할 -> 누수 방지)
# ------------------------------------------------------------------
all_t = list(range(WINDOW - 1, N_DAYS - MAX_H))   # 가능한 기준일: 5 ~ 96

# 학습: 타깃(t+3)까지 모두 학습 구간(<=69일) 안에 있어야 함
train_t = [t for t in all_t if t + MAX_H <= TRAIN_DAYS - 1]
# 테스트: 기준일이 70일 이후 -> 타깃은 전부 테스트 구간 (피처는 과거라 학습 구간을 봐도 무방)
test_t = [t for t in all_t if t >= TRAIN_DAYS]

X_train, Y_train = build_samples(train_t)
X_test, Y_test = build_samples(test_t)
print(f"학습 샘플(기준일) {len(train_t)}개, 테스트 샘플(기준일) {len(test_t)}개")


# ------------------------------------------------------------------
# 4. 공통 유틸: 양성 확률 꺼내기 / H별 평가
# ------------------------------------------------------------------
def pos_proba(classes, proba):
    """predict_proba 결과에서 양성(1) 확률만 반환. 학습에 한 클래스뿐이어도 안전."""
    if 1 in classes:
        return proba[:, list(classes).index(1)]
    return np.zeros(len(proba))


def eval_per_h(y_true, proba, thr=0.5):
    """y_true, proba: (N, MAX_H). 열 j = H=j+1. H별 지표 + 평균 행 반환."""
    rows = []
    for j in range(y_true.shape[1]):
        yt, yp = y_true[:, j], proba[:, j]
        rows.append({
            "H": j + 1,
            "pos_rate": yt.mean(),
            "ROC_AUC": roc_auc_score(yt, yp) if len(np.unique(yt)) > 1 else np.nan,
            "PR_AUC": average_precision_score(yt, yp),
            "F1": f1_score(yt, yp >= thr, zero_division=0),
            "Brier": brier_score_loss(yt, yp),
        })
    df = pd.DataFrame(rows).set_index("H")
    df.loc["평균"] = df.mean()
    return df


# ------------------------------------------------------------------
# 5. 방식 A: 단일 모델 + H를 피처로
# ------------------------------------------------------------------
def to_long(X, Yvec):
    """(n,6),(n,3) -> 기준일당 H=1,2,3 세 행으로 펼침: X_long (3n,7), y_long (3n,)"""
    n = len(X)
    X_rep = np.repeat(X, MAX_H, axis=0)                   # 같은 피처를 3번 복제
    h_col = np.tile(np.arange(1, MAX_H + 1), n)[:, None]  # 1,2,3,1,2,3,...
    return np.hstack([X_rep, h_col]), Yvec.reshape(-1)    # Yvec 행 우선 펼침 = H 순서와 일치


X_tr_long, y_tr_long = to_long(X_train, Y_train)
X_te_long, _ = to_long(X_test, Y_test)

model_a = RandomForestClassifier(n_estimators=200, random_state=SEED)
model_a.fit(X_tr_long, y_tr_long)

proba_long = pos_proba(model_a.classes_, model_a.predict_proba(X_te_long))
proba_a = proba_long.reshape(-1, MAX_H)                  # (N_test, 3)로 복원
res_a = eval_per_h(Y_test, proba_a)

# ------------------------------------------------------------------
# 6. 방식 B: multi-label 벡터 타깃
# ------------------------------------------------------------------
model_b = RandomForestClassifier(n_estimators=200, random_state=SEED)
model_b.fit(X_train, Y_train)                            # Y_train: (n, 3) -> 자동 multi-output

proba_list = model_b.predict_proba(X_test)               # 길이 3 리스트, 각 (N, 클래스수)
proba_b = np.column_stack(
    [pos_proba(cls, p) for cls, p in zip(model_b.classes_, proba_list)]
)                                                        # (N_test, 3)
res_b = eval_per_h(Y_test, proba_b)

# ------------------------------------------------------------------
# 7. 결과 비교
# ------------------------------------------------------------------
pd.set_option("display.float_format", "{:.3f}".format)
print("\n[방식 A] 단일 모델 + H 피처")
print(res_a)
print("\n[방식 B] multi-label 벡터 타깃")
print(res_b)

print("\n[비교] 평균 지표 (A vs B)")
print(pd.DataFrame({"A_H피처": res_a.loc["평균"], "B_벡터타깃": res_b.loc["평균"]})
      .drop("pos_rate"))

# 방식 B 전용 multi-label 지표
pred_b = (proba_b >= 0.5).astype(int)
print(f"\n[B 전용] subset accuracy(3일 모두 정답): {(pred_b == Y_test).all(axis=1).mean():.3f}"
      f" | Hamming loss: {(pred_b != Y_test).mean():.3f}")
pred_a = (proba_a >= 0.5).astype(int)
print(f"[A 동일지표] subset accuracy: {(pred_a == Y_test).all(axis=1).mean():.3f}"
      f" | Hamming loss: {(pred_a != Y_test).mean():.3f}")

전체 양성 비율: 0.44
학습 샘플(기준일) 62개, 테스트 샘플(기준일) 27개

[방식 A] 단일 모델 + H 피처
    pos_rate  ROC_AUC  PR_AUC    F1  Brier
H                                         
1      0.333    0.988   0.977 0.947  0.036
2      0.333    1.000   1.000 1.000  0.023
3      0.333    1.000   1.000 1.000  0.014
평균     0.333    0.996   0.992 0.982  0.025

[방식 B] multi-label 벡터 타깃
    pos_rate  ROC_AUC  PR_AUC    F1  Brier
H                                         
1      0.333    1.000   1.000 1.000  0.026
2      0.333    1.000   1.000 1.000  0.020
3      0.333    1.000   1.000 0.941  0.017
평균     0.333    1.000   1.000 0.980  0.021

[비교] 평균 지표 (A vs B)
         A_H피처  B_벡터타깃
ROC_AUC  0.996   1.000
PR_AUC   0.992   1.000
F1       0.982   0.980
Brier    0.025   0.021

[B 전용] subset accuracy(3일 모두 정답): 0.963 | Hamming loss: 0.012
[A 동일지표] subset accuracy: 0.963 | Hamming loss: 0.012
